In [28]:
#Install the required libraries
%pip install -U langchain langchain_community langchain_classic langchain_chroma langchain_huggingface langchain_ollama chromadb sentence_transformers pypdf streamlit

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
%pip install -U langchain-community

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os

from langchain_community.document_loaders import PyPDFLoader, TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
#Document loading
document_folder = "documents"

documents = []

for filename in os.listdir(document_folder):

    file_path = os.path.join(document_folder, filename)

    if filename.endswith(".pdf"):

        loader = PyPDFLoader(file_path)

        loaded_docs = loader.load()

        documents.extend(loaded_docs)

    elif filename.endswith(".txt"):

        loader = TextLoader(file_path, encoding="utf-8")

        loaded_docs = loader.load()

        documents.extend(loaded_docs)


print("Total documents loaded:", len(documents))

print("\nFirst document content:\n")

print(documents[0].page_content[:1000])

Total documents loaded: 12

First document content:

Position Paper 
Artifcial Intelligence and
Machine Learning 101 
Artifcial intelligence (AI) is transforming the way that we 
interact with machines and the way that machines interact with 
us. This guide breaks down how AI functions, the strengths and 
limitations of various types of machine learning, and the evolution 
of this ever-changing feld of study. It also explores the role of AI-
enabled security analytics or user and entity behavioral analytics 
(UEBA) to better protect enterprises from today’s complex 
cybersecurity threats.


In [ ]:
#Part 3 – Text splitting
#We cannot directly send an entire large document to the LLM. Therefore, we divide it into smaller chunks.
#Cell 4 – Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

print("\nFirst chunk:\n")

print(chunks[0].page_content)

Total chunks created: 35

First chunk:

Position Paper 
Artifcial Intelligence and
Machine Learning 101 
Artifcial intelligence (AI) is transforming the way that we 
interact with machines and the way that machines interact with 
us. This guide breaks down how AI functions, the strengths and 
limitations of various types of machine learning, and the evolution 
of this ever-changing feld of study. It also explores the role of AI-
enabled security analytics or user and entity behavioral analytics 
(UEBA) to better protect enterprises from today’s complex 
cybersecurity threats.


In [9]:
#Load embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model loaded successfully!")

Could not reach the Hub ([Errno 11001] getaddrinfo failed). Using cached commit hash for 'BAAI/bge-small-en-v1.5'.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/2_Normalize/config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/2_Normalize/config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/2_Normalize/config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/2_Normalize/config.json
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/5c38ec7c405ec4b44b94cc5a9bb96e735b3826

Embedding model loaded successfully!


In [10]:
#Generate embeddings
sample_text = chunks[0].page_content

sample_embedding = embedding_model.embed_query(sample_text)

print("Original text:")
print(sample_text[:200])

print("\nEmbedding:")
print(sample_embedding[:10])

print("\nEmbedding dimensions:")
print(len(sample_embedding))

Original text:
Position Paper 
Artifcial Intelligence and
Machine Learning 101 
Artifcial intelligence (AI) is transforming the way that we 
interact with machines and the way that machines interact with 
us. This g

Embedding:
[-0.0020789150148630142, -0.031145721673965454, 0.0038918014615774155, -0.05308705195784569, 0.051137883216142654, -0.010263395495712757, 0.051659613847732544, 0.0042732455767691135, 0.01177307777106762, -0.008602463640272617]

Embedding dimensions:
384


In [11]:
#Part 5 – Vector database using ChromaDB
#Now we store our document chunks and their embeddings inside ChromaDB.
#Cell 7 – Create ChromaDB vector database
import shutil

vector_db_path = "./chroma_db"

# Remove old database when rebuilding the project
if os.path.exists(vector_db_path):
    shutil.rmtree(vector_db_path)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=vector_db_path,
    collection_name="rag_documents"
)

print("Documents successfully stored in ChromaDB!")

Documents successfully stored in ChromaDB!


In [12]:
#Cell 8 – Test similarity search
#Before integrating the LLM, let's check whether our vector database can retrieve relevant document chunks.
query = "What is the main topic of this document?"

retrieved_docs = vectorstore.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(retrieved_docs):

    print(f"\nDocument Chunk {i+1}")

    print(doc.page_content[:500])

    print("Metadata:", doc.metadata)

    print("-" * 50)


Document Chunk 1
Artificial Intelligence and Machine Learning 101
 
 
   
 
   
  
  
Picture stopping at a roadway intersection  Y our eyes see that the trafc light in front of 
you has just turned green  Based on what you have learned from experience (and driver’s 
education), you know that a green light indicates that you should drive forward  So, you hit 
the gas pedal  The green light is the raw input, your acceleration is the output; everything in 
between is processing  
To intelligently navigate the worl
Metadata: {'keywords': 'Machine Learn; Human Leanring; Security; Interest', 'page': 3, 'producer': 'Adobe PDF Library 17.0', 'creationdate': '2024-01-25T09:51:12-06:00', 'trapped': '/False', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'total_pages': 12, 'author': 'OpenText Cybersecurity', 'moddate': '2024-01-26T15:07:00-08:00', 'source': 'documents\\aiandml.pdf', 'subject': 'Artificial intelligence (AI) is transforming the way that we interact with machines and the way that 

In [ ]:
#Part 6 – Integrate the LLM using Ollama
#Now comes the generation part of RAG.

#We will use TinyLlama through Ollama.
llm = ChatOllama(
    model="tinyllama",
    temperature=0.2
)

print("Ollama LLM initialized successfully!")


Ollama LLM initialized successfully!


In [17]:
response = llm.invoke(
    "Explain what Artificial Intelligence is in simple words."
)

print(response.content)

Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machines that can perform tasks that are normally done by humans. In simpler words, AI is a system that mimics the way humans think and behave.

AI is divided into two main categories: Natural Language Processing (NLP) and Machine Learning (ML). NLP is concerned with understanding human language, while ML is concerned with learning from data and making predictions based on it.

AI is becoming increasingly important in various fields, such as healthcare, finance, and manufacturing. It has the potential to revolutionize these industries by improving efficiency, reducing costs, and enabling new business models.

Overall, AI is a powerful tool that can help solve complex problems and improve our lives.


Part 7 – Create the RAG chain using LangChain

In [18]:
#Cell 11 – Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3
    }
)

print("Retriever created successfully!")

Retriever created successfully!


In [19]:
#Cell 12 – Create prompt template
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the user's question using ONLY the
provided context.

If the answer is not available in the context,
clearly say:

"I could not find the answer in the uploaded documents."

Do not make up information.

Context:
{context}

Question:
{input}

Answer:
""")

print("Prompt template created!")

Prompt template created!


In [20]:
#Cell 13 – Create document processing chain
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

print("Document chain created successfully!")

Document chain created successfully!


In [21]:
#Cell 14 – Create complete retrieval chain
rag_chain = create_retrieval_chain(
    retriever,
    document_chain
)

print("Complete RAG chain created successfully!")

Complete RAG chain created successfully!


Part 8 – Ask questions to your chatbot
Now we can finally interact with our RAG system.

In [22]:
question = "What is the main topic discussed in the document?"

response = rag_chain.invoke({
    "input": question
})

print("Question:")
print(question)

print("\nAnswer:")
print(response["answer"])

Question:
What is the main topic discussed in the document?

Answer:
The main topic discussed in the document is the process of human intelligence, which includes the three general steps of input, processing, and output. The document also covers the three aspects of human intelligence, including knowledge and memory, decision and inference, and learning. The document provides examples of how these aspects work in the context of human intelligence, such as how knowledge is formed, how memory is used, and how decision-making and inference are made based on raw input.


In [23]:
#Cell 16 – Display retrieved source documents
print("Retrieved Source Documents:\n")

for i, doc in enumerate(response["context"]):

    print(f"Source {i+1}")

    print("Content:")
    print(doc.page_content[:500])

    print("\nMetadata:")
    print(doc.metadata)

    print("=" * 60)

Retrieved Source Documents:

Source 1
Content:
we process it, and ultimately the information helps us act  
Let’s break this down into a system diagram  In the fgure below, the three general steps 
of human intelligence from left to right: input, processing, and output  In the human brain, 
input takes place in the form of sensing and perceiving things  Y our eyes, nose, ears, etc , 
take in raw input on the left, such as photons of light or the smell of pine trees, and then 
process it  On the system’s right side is output  This includes sp

Metadata:
{'moddate': '2024-01-26T15:07:00-08:00', 'title': 'Artificial Intelligence and Machine Learning 101 | Open Text', 'creationdate': '2024-01-25T09:51:12-06:00', 'page': 2, 'total_pages': 12, 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'trapped': '/False', 'keywords': 'Machine Learn; Human Leanring; Security; Interest', 'page_label': '1', 'producer': 'Adobe PDF Library 17.0', 'source': 'documents\\aiandml.pdf', 'author': 'OpenText Cyberse

In [24]:
#Cell 17 – Create an interactive notebook chatbot
while True:

    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Chatbot stopped.")
        break

    response = rag_chain.invoke({
        "input": question
    })

    print("\nAI Answer:")
    print(response["answer"])

    print("\nSources:")

    for doc in response["context"]:
        print(doc.metadata.get("source", "Unknown source"))


AI Answer:
Answer the user's question using ONLY the provided context.

If the answer is not available in the context,
clearly say:

"I could not find the answer in the uploaded documents."

Do not make up information.

Sources:
documents\aiandml.pdf
documents\aiandml.pdf
documents\aiandml.pdf

AI Answer:
Yes, the context provided in the provided text is the topic of AI and machine learning 101. The guide covers the basics of AI, including the functions and strengths of various types of machine learning, as well as the evolution of this ever-changing field of study. The guide also explores the role of AI in cybersecurity, with a focus on UEBA (user and entity behavioral analytics) to better protect enterprise systems. The main question asked is "what is Artificial Intellegence?"

Sources:
documents\aiandml.pdf
documents\aiandml.pdf
documents\aiandml.pdf


KeyboardInterrupt: Interrupted by user

Part 9 – Build the Streamlit user interface

In [27]:
%%writefile app.py

import os
import tempfile
import hashlib
import streamlit as st

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


# -----------------------------------
# PAGE CONFIGURATION
# -----------------------------------

st.set_page_config(
    page_title="RAG AI Chatbot",
    page_icon="🤖",
    layout="wide"
)

st.title("🤖 Document-Based AI Chatbot")

st.write(
    "Upload your documents and ask questions "
    "based on their content using RAG."
)


# -----------------------------------
# SESSION STATE
# -----------------------------------

if "rag_chain" not in st.session_state:
    st.session_state.rag_chain = None

if "messages" not in st.session_state:
    st.session_state.messages = []

if "file_hash" not in st.session_state:
    st.session_state.file_hash = None


# -----------------------------------
# LOAD MODELS
# -----------------------------------

@st.cache_resource
def load_embeddings():

    return HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )


@st.cache_resource
def load_llm():

    return ChatOllama(
        model="tinyllama",
        temperature=0.2
    )


# -----------------------------------
# DOCUMENT PROCESSING
# -----------------------------------

def process_documents(uploaded_files):

    documents = []

    for uploaded_file in uploaded_files:

        suffix = os.path.splitext(uploaded_file.name)[1].lower()

        if suffix not in [".pdf", ".txt"]:
            continue

        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=suffix
        ) as temp_file:

            temp_file.write(uploaded_file.getvalue())

            temp_path = temp_file.name

        try:

            if suffix == ".pdf":

                loader = PyPDFLoader(temp_path)

            else:

                loader = TextLoader(
                    temp_path,
                    encoding="utf-8"
                )

            loaded_documents = loader.load()

            for doc in loaded_documents:

                doc.metadata["source"] = uploaded_file.name

            documents.extend(loaded_documents)

        finally:

            if os.path.exists(temp_path):
                os.remove(temp_path)

    if not documents:
        raise ValueError("No readable PDF or TXT documents found.")

    return documents


# -----------------------------------
# CREATE VECTOR DATABASE AND RAG CHAIN
# -----------------------------------

def create_rag_system(documents):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(documents)

    if not chunks:
        raise ValueError("No text could be extracted from the documents.")

    embeddings = load_embeddings()

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="rag_" + os.urandom(8).hex()
    )

    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}
    )

    llm = load_llm()

    prompt = ChatPromptTemplate.from_template("""
    You are a helpful document-based AI assistant.

    Answer only using the supplied context.

    If the answer is not available in the context,
    say that you could not find the answer in the documents.

    Do not invent information.

    Context:
    {context}

    Question:
    {input}

    Answer:
    """)

    document_chain = create_stuff_documents_chain(
        llm,
        prompt
    )

    rag_chain = create_retrieval_chain(
        retriever,
        document_chain
    )

    return rag_chain, len(chunks)


# -----------------------------------
# SIDEBAR
# -----------------------------------

st.sidebar.header("📁 Upload Documents")

uploaded_files = st.sidebar.file_uploader(
    "Upload PDF or TXT files",
    type=["pdf", "txt"],
    accept_multiple_files=True
)

if st.sidebar.button("Process Documents", type="primary"):

    if not uploaded_files:

        st.sidebar.warning("Please upload documents first.")

    else:

        try:

            current_hash = hashlib.md5(
                b"".join(
                    f.name.encode() + f.getvalue()
                    for f in uploaded_files
                )
            ).hexdigest()

            if current_hash == st.session_state.file_hash:

                st.sidebar.info("These documents are already processed.")

            else:

                with st.spinner("Loading and processing documents..."):

                    documents = process_documents(uploaded_files)

                    rag_chain, chunk_count = create_rag_system(
                        documents
                    )

                    st.session_state.rag_chain = rag_chain
                    st.session_state.file_hash = current_hash
                    st.session_state.messages = []

                st.sidebar.success(
                    f"Successfully processed {len(documents)} pages/records "
                    f"into {chunk_count} chunks!"
                )

        except Exception as e:

            st.sidebar.error(f"Processing failed: {str(e)}")


if st.sidebar.button("Clear Chat"):

    st.session_state.messages = []

    st.rerun()


# -----------------------------------
# DISPLAY CHAT HISTORY
# -----------------------------------

for message in st.session_state.messages:

    with st.chat_message(message["role"]):

        st.markdown(message["content"])

        if message.get("sources"):

            with st.expander("View source documents"):

                for source in message["sources"]:
                    st.write(source)


# -----------------------------------
# CHAT INPUT
# -----------------------------------

question = st.chat_input(
    "Ask something about your uploaded documents..."
)

if question:

    if st.session_state.rag_chain is None:

        st.warning(
            "Please upload and process documents before asking questions."
        )

    else:

        st.session_state.messages.append({
            "role": "user",
            "content": question
        })

        with st.chat_message("user"):
            st.markdown(question)

        with st.chat_message("assistant"):

            with st.spinner("Searching documents and generating answer..."):

                try:

                    response = st.session_state.rag_chain.invoke({
                        "input": question
                    })

                    answer = response["answer"]

                    sources = []

                    for doc in response["context"]:

                        source_name = doc.metadata.get(
                            "source", "Unknown source"
                        )

                        page_number = doc.metadata.get("page")

                        source_info = (
                            f"{source_name} | Page {page_number + 1}"
                            if page_number is not None
                            else source_name
                        )

                        if source_info not in sources:
                            sources.append(source_info)

                    st.markdown(answer)

                    with st.expander("View source documents"):

                        for source in sources:
                            st.write("📄", source)

                    st.session_state.messages.append({
                        "role": "assistant",
                        "content": answer,
                        "sources": sources
                    })

                except Exception as e:

                    st.error(f"An error occurred: {str(e)}")

Writing app.py
